In [1]:
import geopandas as gpd
import pandas as pd
import polars as pl

In [ ]:
demanda_scan = pl.scan_parquet(r"../outputs/01/Dados_4Meses_2023_2024.parquet")
demanda_lazy = demanda_scan.select(["data", "linha_blt", "zona_emb"])  # Pega só o que precisa

In [3]:
demanda_lazy = demanda_lazy.with_columns(
    pl.col("data").str.slice(0, 4).alias("Ano"),
    pl.col("data").str.slice(4, 2).alias("Mês")
)

In [4]:
demanda_mes = (
    demanda_lazy
    .group_by(["linha_blt", "zona_emb", "Ano", "Mês"])
    .agg(
        pl.len().alias("N_embarques")
    )
    .collect(engine="streaming")  # roda tudo (leitura + filtro + agregação) em streaming, sem materializar a base inteira
)

In [5]:
demanda_mes

linha_blt,zona_emb,Ano,Mês,N_embarques
cat,f64,str,str,u32
"""514T-10""",261.0,"""2023""","""04""",36292
"""2719-10""",181.0,"""2023""","""04""",31950
"""5110-10""",18.0,"""2023""","""04""",27301
"""971R-10""",123.0,"""2023""","""04""",34535
"""1024-10""",125.0,"""2023""","""04""",36518
…,…,…,…,…
"""4008-10""",234.0,"""2024""","""10""",6
"""128Y-10""",154.0,"""2024""","""10""",5
"""978A-10""",124.0,"""2024""","""10""",23


In [ ]:
demanda_mes.write_parquet("../outputs/01/Dados_4Meses_2023_2024_mes.parquet")